In [1]:
import json, glob, os
import pandas as pd

files = sorted(glob.glob("out/*/data.json"))
print("data.json files:", len(files))

rows = []
samples = []

for fp in files:
    sample = os.path.basename(os.path.dirname(fp))
    samples.append(sample)
    with open(fp, "r") as f:
        data = json.load(f)

    ent = (data.get("plasmidfinder", {})
              .get("results", {})
              .get("Enterobacteriales", {})
              .get("enterobacteriales", {}))

    if isinstance(ent, dict) and ent:
        for hit_id, hit in ent.items():
            replicon_allele = ""
            parts = hit_id.split(":")
            if len(parts) >= 2:
                replicon_allele = parts[-2]

            rows.append({
                "Sample": sample,
                "PlasmidFamily": hit.get("plasmid", ""),
                "RepliconAllele": replicon_allele,
                "Accession": hit.get("accession", ""),
                "Identity": hit.get("identity", None),
                "Coverage": hit.get("coverage", None),
                "HSP_length": hit.get("HSP_length", None),
                "Template_length": hit.get("template_length", None),
                "Positions_in_contig": hit.get("positions_in_contig", ""),
                "Contig_name": hit.get("contig_name", ""),
                "Note": hit.get("note", ""),
                "Hit_id": hit_id
            })

df_long = pd.DataFrame(rows)
samples = sorted(set(samples))

# Summary
if df_long.empty:
    df_summary = pd.DataFrame({"Sample": samples})
    df_summary["replicon_hits"] = 0
    df_summary["unique_replicon_families"] = 0
    df_summary["replicon_families"] = ""
    df_wide = pd.DataFrame(index=samples)
else:
    fams = (df_long.groupby("Sample")["PlasmidFamily"]
            .apply(lambda x: sorted(set([v for v in x if v]))))
    df_summary = pd.DataFrame({"Sample": samples}).merge(
        fams.rename("replicon_families_list"),
        left_on="Sample", right_index=True, how="left"
    )
    df_summary["replicon_families_list"] = df_summary["replicon_families_list"].apply(
        lambda v: v if isinstance(v, list) else []
    )
    df_summary["replicon_hits"] = df_summary["Sample"].map(df_long.groupby("Sample").size()).fillna(0).astype(int)
    df_summary["unique_replicon_families"] = df_summary["replicon_families_list"].apply(len)
    df_summary["replicon_families"] = df_summary["replicon_families_list"].apply(lambda v: ",".join(v))
    df_summary = df_summary.drop(columns=["replicon_families_list"])

    # Presence/absence matrix
    df_wide = (df_long.assign(Present=1)
               .pivot_table(index="Sample", columns="PlasmidFamily", values="Present",
                            aggfunc="max", fill_value=0)
               .reindex(samples, fill_value=0))
    df_wide.index.name = "Sample"

# Write outputs
df_long.to_csv("plasmidfinder_replicons_long.tsv", sep="\t", index=False)
df_wide.to_csv("plasmidfinder_replicons_wide.tsv", sep="\t")
df_summary.to_csv("plasmidfinder_summary.tsv", sep="\t", index=False)

df_summary.head()

data.json files: 64


,Sample,replicon_hits,unique_replicon_families,replicon_families
0,GCA_002038285.1,2,2,"IncFIB(S),IncFII(S)"
1,GCA_002038445.1,2,2,"IncFIB(S),IncFII(S)"
2,GCA_002038485.1,2,2,"IncFIB(S),IncFII(S)"
3,GCA_002038585.1,2,2,"IncFIB(S),IncFII(S)"
4,GCA_002038665.1,2,2,"IncFIB(S),IncFII(S)"


In [9]:
df_wide

PlasmidFamily,IncFIB(S),IncFII(S),IncI1-I(Alpha),IncI2(Delta),IncX1
Sample,,,,,
GCA_002038285.1,1,1,0,0,0
GCA_002038445.1,1,1,0,0,0
GCA_002038485.1,1,1,0,0,0
GCA_002038585.1,1,1,0,0,0
GCA_002038665.1,1,1,0,0,0
...,...,...,...,...,...
GCA_037003885.1,0,0,0,0,1
GCA_041411215.1,1,1,0,0,1
GCA_045376435.1,1,1,0,0,1
